# Questão 7 — Sistema de Recomendação

## Objetivo

Construir um sistema simples de recomendação item-item baseado no comportamento
de compra dos clientes.

O produto de referência é `Motor de Popa 1949`. A recomendação será baseada
na Similaridade de Cosseno entre produtos, utilizando uma matriz binária de
interações Cliente × Produto.

### Estratégia

1. Relacionar pedidos, itens, variantes e produtos;
2. Criar uma interação única para cada par cliente-produto;
3. Construir uma matriz binária Cliente × Produto;
4. Transpor a matriz para comparar produtos pelos clientes que os compraram;
5. Calcular a Similaridade de Cosseno;
6. Remover o próprio produto de referência;
7. Selecionar os 5 produtos com maior similaridade.

In [ ]:
from pathlib import Path

import duckdb
import numpy as np
import pandas as pd


PROJECT_ROOT = Path.cwd().parent
RAW_DATA_DIR = PROJECT_ROOT / "data" / "raw"
OUTPUT_DIR = PROJECT_ROOT / "data" / "outputs"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

TARGET_PRODUCT_NAME = "Motor de Popa 1949"

conn = duckdb.connect()

In [ ]:
tables = [
    "products",
    "product_variants",
    "orders",
    "order_items",
]

for table in tables:
    csv_path = RAW_DATA_DIR / f"{table}.csv"

    conn.execute(
        f"""
        CREATE OR REPLACE TABLE {table} AS
        SELECT *
        FROM read_csv_auto('{csv_path.as_posix()}');
        """
    )

print("Tabelas carregadas com sucesso.")

Tabelas carregadas com sucesso.


In [ ]:
query_interactions = """
SELECT DISTINCT
    o.customer_id,
    pv.product_id,
    1 AS interaction
FROM orders AS o
INNER JOIN order_items AS oi
    ON oi.order_id = o.id
INNER JOIN product_variants AS pv
    ON pv.id = oi.product_variant_id
INNER JOIN products AS p
    ON p.id = pv.product_id
WHERE o.customer_id IS NOT NULL;
"""

interactions = conn.execute(query_interactions).df()

interactions.head()

,customer_id,product_id,interaction
0,1998,155,1
1,1591,156,1
2,1109,113,1
3,930,331,1
4,1585,284,1


In [ ]:
user_product_matrix = interactions.pivot_table(
    index="customer_id",
    columns="product_id",
    values="interaction",
    aggfunc="max",
    fill_value=0,
).astype("uint8")

user_product_matrix

product_id,1,2,3,4,5,6,7,8,9,10,...,491,492,493,494,495,496,497,498,499,500
customer_id,,,,,,,,,,,,,,,,,,,,,
1,0,1,0,0,0,0,0,0,0,0,...,0,1,0,0,0,0,0,0,0,0
2,0,0,0,0,0,0,0,0,1,0,...,0,0,1,0,0,0,0,0,0,0
3,0,0,0,0,0,1,0,0,1,0,...,0,0,0,0,0,0,0,0,0,0
4,0,0,0,0,1,0,1,1,0,1,...,0,1,0,0,1,0,0,0,0,0
5,0,0,1,0,0,1,0,0,1,0,...,0,1,0,0,0,0,1,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1996,0,0,0,0,1,0,1,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1997,1,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1998,0,0,0,0,0,0,0,0,1,0,...,0,0,0,0,0,0,1,0,0,0


In [ ]:
print(f"Clientes: {user_product_matrix.shape[0]}")
print(f"Produtos: {user_product_matrix.shape[1]}")

Clientes: 2000
Produtos: 500


In [ ]:
target_product = conn.execute(
    """
    SELECT
        id,
        name
    FROM products
    WHERE name = ?
    """,
    [TARGET_PRODUCT_NAME],
).df()

target_product

,id,name
0,180,Motor de Popa 1949


In [ ]:
if len(target_product) != 1:
    raise ValueError(
        f"Expected exactly one product named '{TARGET_PRODUCT_NAME}', "
        f"but found {len(target_product)}."
    )

target_product_id = int(target_product.iloc[0]["id"])

print(f"Produto de referência: {TARGET_PRODUCT_NAME}")
print(f"Product ID: {target_product_id}")

Produto de referência: Motor de Popa 1949
Product ID: 180


In [ ]:
product_user_matrix = user_product_matrix.T.astype(float)

product_user_matrix.shape

(500, 2000)

In [ ]:
target_vector = product_user_matrix.loc[
    target_product_id
].to_numpy()

product_vectors = product_user_matrix.to_numpy()

dot_products = product_vectors @ target_vector

product_norms = np.linalg.norm(
    product_vectors,
    axis=1,
)

target_norm = np.linalg.norm(target_vector)

denominator = product_norms * target_norm

similarities = np.divide(
    dot_products,
    denominator,
    out=np.zeros_like(dot_products, dtype=float),
    where=denominator != 0,
)

In [ ]:
similarity_ranking = pd.DataFrame(
    {
        "product_id": product_user_matrix.index,
        "similaridade": similarities,
    }
)

In [ ]:
similarity_ranking = similarity_ranking[
    similarity_ranking["product_id"] != target_product_id
]

In [ ]:
product_names = conn.execute(
    """
    SELECT
        id AS product_id,
        name AS produto
    FROM products;
    """
).df()

In [ ]:
similarity_ranking = similarity_ranking.merge(
    product_names,
    on="product_id",
    how="left",
)

In [ ]:
top_5 = (
    similarity_ranking
    .sort_values(
        by=["similaridade", "product_id"],
        ascending=[False, True],
    )
    .head(5)
    .reset_index(drop=True)
)

top_5

top_5.to_csv(
    OUTPUT_DIR / "product_recommendations.csv",
    index=False,
)

print("product_recommendations.csv exportado com sucesso.")

,product_id,similaridade,produto
0,389,0.256553,Motor de Popa 5331
1,295,0.256239,Cabo Náutico 2105
2,75,0.255785,Vela Mestra 1913
3,337,0.239332,Cabo Náutico 9048
4,55,0.237744,GPS Plotter 6249


In [ ]:
target_buyers = int(
    user_product_matrix[target_product_id].sum()
)

print(
    f"Clientes que compraram '{TARGET_PRODUCT_NAME}': "
    f"{target_buyers}"
)

Clientes que compraram 'Motor de Popa 1949': 397


In [ ]:
unique_values = np.unique(user_product_matrix.to_numpy())

print(f"Valores presentes na matriz: {unique_values}")

assert set(unique_values).issubset({0, 1})

Valores presentes na matriz: [0 1]
